# Imports

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

from sko.PSO import PSO
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from scipy.stats import qmc
from tqdm.auto import tqdm

import finesse
from finesse.knm import Map
from finesse.utilities.maps import circular_aperture

In [ ]:
# Import from parent directory
from pathlib import Path
import sys
current_dir = Path.cwd()  
parent_dir = current_dir.parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Import kat script
import utils.finesse_base as finesse_model
def make_model():
    return finesse_model.base_kat.deepcopy()

# Finesse Wrappers

In [ ]:
# aperture surface maps
grid = np.linspace(-0.17, 0.17, 601)
aperture = circular_aperture(grid, grid, 0.17)
tm_map = Map(grid, grid, amplitude=aperture)

# inputs: itm roc, etm roc, input q param
# outputs: circ power, locked input q param
# default maxtem=6
def run_finesse(itm_roc, etm_roc, q_in=None, maxtem=6):
    model = make_model()
    model.lock_length.accuracy=1e-9
    model.modes(maxtem=maxtem, modes='even')
    model.ITM.Rc = itm_roc
    model.ETM.Rc = etm_roc
    if q_in is not None and q_in != 0:
        model.parse(f'gauss q_in ITM.p1.i q={q_in}')
    model.parse("bp q_det ITM.p1.i q")
    # stability check
    if not model.cavArm.is_stable:
        return np.nan, complex(np.nan, np.nan)
    # apply maps
    model.ITM.surface_map = tm_map
    model.ETM.surface_map = tm_map
    # run
    try:
        out = model.run("series(run_locks(max_iterations=50000), noxaxis())")["noxaxis"]
        return out['circ'], out['q_det']
    except Exception as e:
        print(e)
        return np.nan, complex(np.nan, np.nan)

# outer function for computing tolerances to itm and etm perturbations
def run_finesse_perturb(itm_roc, etm_roc, q_in=None, maxtem=6, delta_roc=0.33):
    pows = np.zeros(4)
    pow_nom, q_nom = run_finesse(itm_roc, etm_roc, q_in, maxtem)
    if not np.isfinite(pow_nom):
        return np.nan, np.nan, np.nan, complex(np.nan, np.nan)

    itm_perturb = np.array([1, -1, 0, 0]) * delta_roc /100
    etm_perturb = np.array([0, 0, 1, -1]) * delta_roc /100
    for i in range(len(itm_perturb)):
        pow, _ = run_finesse(itm_roc*(1+itm_perturb[i]), etm_roc*(1+etm_perturb[i]), q_nom, maxtem)
        pows[i] = pow
    
    tol_itm = np.abs((pows[0]-pow_nom)/(itm_roc*itm_perturb[0])) + np.abs((pows[1]-pow_nom)/(itm_roc*itm_perturb[1]))
    tol_etm = np.abs((pows[2]-pow_nom)/(etm_roc*etm_perturb[2])) + np.abs((pows[3]-pow_nom)/(etm_roc*etm_perturb[3]))

    return pow_nom, tol_itm, tol_etm, q_nom

Cost functions

In [ ]:
def calc_cost_power(X):
    itm_roc, etm_roc = X[0], X[1]
    circ, _ = run_finesse(itm_roc, etm_roc)
    cost = 1/circ
    return cost

def calc_cost_perturb(X):
    itm_roc, etm_roc = X[0], X[1]
    p_circ, tol_itm, tol_etm, _ = run_finesse_perturb(itm_roc, etm_roc)
    cost = 1/p_circ + tol_itm + tol_etm
    return cost

# Heat Map

In [ ]:
# heat map
resolution = 3
itm_roc = np.linspace(-4134, -1434, resolution)
etm_roc = np.linspace(1745, 4245, resolution)
i_r, e_r = np.meshgrid(itm_roc, etm_roc)

vect_run = np.vectorize(run_finesse)
results = vect_run(i_r, e_r)

In [ ]:
p_circ, _ = results

fig, ax = plt.subplots(figsize=(8, 6))
heatmap = ax.pcolormesh(i_r, e_r, p_circ, cmap="viridis", shading="auto", vmin=282)
cbar = fig.colorbar(heatmap, ax=ax)
cbar.set_label("Circulating Power (W)", fontsize=12)
ax.set_title("Circulating Power over ROC Space", fontsize=14, pad=15)
ax.set_xlabel("ITM ROC (m)", fontsize=12)
ax.set_ylabel("ETM ROC (m)", fontsize=12)

## Cost Heatmap

In [ ]:
resolution = 20
itm_roc = np.linspace(-4134, -1434, resolution)
etm_roc = np.linspace(1745, 4245, resolution)
i_r, e_r = np.meshgrid(itm_roc, etm_roc)

vect_run = np.vectorize(run_finesse_perturb)
results = vect_run(i_r, e_r)
p_circ, tol_itm, tol_etm, _ = results
cost = 1/p_circ + tol_itm + tol_etm

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
heatmap = ax.pcolormesh(i_r, e_r, np.log10(cost), cmap="viridis", shading="auto", vmax=-1.5)
cbar = fig.colorbar(heatmap, ax=ax)
cbar.set_label("Cost (Log Scaled)", fontsize=12)
ax.set_title("Cost over ROC Space", fontsize=14, pad=15)
ax.set_xlabel("ITM ROC (m)", fontsize=12)
ax.set_ylabel("ETM ROC (m)", fontsize=12)

# PSO Test

In [ ]:
class FixedPSO(PSO):
    # Fix from SLAC-ML/ligo-opt: keep gbest_x paired with gbest_y.
    def update_gbest(self):
        i = self.pbest_y.argmin()
        if self.gbest_y > self.pbest_y[i]:
            self.gbest_x = self.pbest_x[i].copy()
            self.gbest_y = self.pbest_y[i].copy()

n_evals = 0
def objective(x):
    global n_evals
    n_evals += 1
    return calc_cost_perturb(x)

pso = FixedPSO(
    func=objective, n_dim=2,
    pop=15, max_iter=30,
    lb=[-4134, 1745], ub=[-1434, 4245],
    w=0.9, c1=2.0, c2=0.4,
)

# The initial swarm is evaluated in __init__; make sure it can count as a best.
pso.update_pbest()
pso.update_gbest()

pso.run()

print("best design:", pso.gbest_x, "cost:", pso.gbest_y)
print(f"objective calls: {n_evals}")

Plot result

In [ ]:
p_circ, tol_itm, tol_etm, _ = results # assumes the cost heatmap was run
cost = 1/p_circ + tol_itm + tol_etm

fig, ax = plt.subplots(figsize=(8, 6))
heatmap = ax.pcolormesh(i_r, e_r, np.log10(cost), cmap="viridis", shading="auto", vmax=-1.5)
ax.plot(pso.gbest_x[0], pso.gbest_x[1], 'ro', label=f'PSO Best, Cost={np.log10(pso.gbest_y[0]):.5f}')

best_idx = np.nanargmin(cost)
row, col = np.unravel_index(best_idx, cost.shape)
ax.plot(i_r[row][col], e_r[row][col], 'bo', label=f'Sweep Best, Cost={np.log10(cost[row][col]):.5f}')
cbar = fig.colorbar(heatmap, ax=ax)
cbar.set_label("Circulating Power (W)", fontsize=12)
ax.set_title("PSO Result", fontsize=14, pad=15)
ax.set_xlabel("ITM ROC (m)", fontsize=12)
ax.set_ylabel("ETM ROC (m)", fontsize=12)
ax.legend()
plt.show()

# Neural Network Training

In [ ]:
import torch
from torch import nn
from scipy.stats import qmc

def nn_model(in_dim, hidden_dim, out_dim):
    model = nn.Sequential(
        nn.Linear(in_dim, hidden_dim), nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
        nn.Linear(hidden_dim, out_dim)
    )
    return model

def fit(X, Y, model, epochs=1000, lr=1e-3):
    xm, xs, ym, ys = X.mean(0), X.std(0), Y.mean(0), Y.std(0)
    Xt = torch.tensor((X - xm) / xs, dtype=torch.float32)
    Yt = torch.tensor((Y - ym) / ys, dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = ((model(Xt) - Yt) ** 2).mean() # MSE
        loss.backward()
        opt.step()
    
    def predict(Xq):
        with torch.no_grad():
            out = model(torch.tensor((Xq - xm) / xs, dtype=torch.float32)).numpy()
            return out * ys + ym
    return predict, loss.item()

In [ ]:
def stable(itm_roc, etm_roc, margin=0.05):
    L_CAV = 3994.47 # hardcoded
    g1 = 1 + L_CAV / itm_roc      # check sign convention against model.cavArm.is_stable
    g2 = 1 - L_CAV / etm_roc
    return (g1 * g2 > margin) & (g1 * g2 < 1-margin)

def run(n_init=30, n_trials=50, grid_n=300, seed=0):
    rng = np.random.default_rng(seed)

    LB, UB = np.array([-4134., 1745.]), np.array([-1434., 4245.])
    X, Y = [], []

    for x in qmc.scale(qmc.LatinHypercube(d=2, seed=seed).random(n_init), LB, UB):
        y = calc_cost_perturb(x)
        if np.isfinite(y):
            X.append(x)
            Y.append(y)
    X, Y = np.array(X), np.array(Y).reshape(-1, 1)

    # create a list of valid grid points to evaluate the model on (stable)
    itm_roc = np.linspace(-4134, -1434, grid_n)
    etm_roc = np.linspace(1745, 4245, grid_n)
    i_r, e_r = np.meshgrid(itm_roc, etm_roc)
    g = stable(i_r, e_r)
    grid = np.column_stack([i_r[g], e_r[g]])   

    history = []
    predict_model = None
    for t in range(n_trials):
        torch.manual_seed(seed + t)
        # Create nn
        model = nn_model(in_dim=2, hidden_dim=64, out_dim=1)

        # predicted best
        predict, train_loss = fit(X,np.log(Y),model)
        if t == n_trials-1:
            predict_model = predict

        cost_pred = np.exp(predict(grid))
        x_opt_pred = grid[np.nanargmin(cost_pred)]

        # if optimal point has already been selected, random select another point on the grid
        if np.min(np.linalg.norm((X - x_opt_pred) / (UB - LB), axis=1)) < 1e-3:
            x_opt_pred = grid[rng.integers(len(grid))]
        
        y_opt = calc_cost_perturb(x_opt_pred)
        if not np.isfinite(y_opt):
            print(f"{t:3d}  FAILED at {x_opt_pred}")
            grid = grid[np.any(grid != x_opt_pred, axis=1)]
            continue # if bad point, throw out

        # Append new point
        X, Y = np.vstack([X, x_opt_pred]), np.vstack([Y, y_opt])
        history.append({"round": t, "x": x_opt_pred, "cost_true": y_opt,"cost_pred": np.nanmin(cost_pred), "train_loss": train_loss})
        print(f"{t:3d}  true={history[-1]['cost_true']:.5g}  "
              f"pred={history[-1]['cost_pred']:.5g}")
    return X, Y, history, predict_model

In [ ]:
torch.set_num_threads(1)        # tiny model: one thread avoids contention overhead
torch.set_flush_denormal(True)  # treat subnormal floats as zero
xs, ys, history, model = run(n_init=100, n_trials=100, grid_n=500, seed=0)

In [ ]:
resolution = 50
itm_roc = np.linspace(-4134, -1434, resolution)
etm_roc = np.linspace(1745, 4245, resolution)
i_r, e_r = np.meshgrid(itm_roc, etm_roc)

# Evaluate the NN on the full mesh and mask the unstable region
pts = np.column_stack([i_r.ravel(), e_r.ravel()])
log_pred = np.asarray(model(pts)).reshape(i_r.shape)   # natural log of cost

fig, ax = plt.subplots(figsize=(7, 5.5))
pc = ax.pcolormesh(i_r, e_r, log_pred, cmap="viridis", shading="auto", vmax=1)
fig.colorbar(pc, ax=ax, label="NN predicted cost (log scaled)")

ax.set_xlabel("ITM ROC (m)"); ax.set_ylabel("ETM ROC (m)")
ax.set_title("NN prediction")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(np.min(log_pred))

In [ ]:
rounds = np.array([h["round"] for h in history])
costs = np.array([h["cost_true"] for h in history]) / 2 # to match ma's cost convention
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rounds, costs, "o", alpha=0.4, label="cost at proposal")
ax.plot(rounds, np.minimum.accumulate(costs), "-", lw=2, label="best so far")
ax.axhline(pso.gbest_y[0] / 2, c='r', ls='--', label='PSO Optimal')
ax.set_yscale("log")
ax.set_xlabel("Round")
ax.set_ylabel("Cost")
ax.legend()
ax.set_title("Proposed Best Cost vs. PSO Optimum")
plt.show()

# predict vs. true
rounds = np.array([h["round"] for h in history])
predicts = np.array([h["cost_pred"] for h in history]) / 2
costs = np.array([h["cost_true"] for h in history]) / 2
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rounds, predicts, "-", alpha=0.4, label="Predicted")
ax.plot(rounds, costs, "-", lw=2, label="Actual")
ax.axhline(pso.gbest_y[0] /2 , c='r', ls='--', label='PSO Optimal')
ax.set_yscale("log")
ax.set_xlabel("Round")
ax.set_ylabel("Cost")
ax.set_title("Proposed Cost vs. Actual Cost")
ax.legend()
plt.show()

# error
fig, ax = plt.subplots(figsize=(6, 4))
rounds = np.array([h["round"] for h in history])
predicts = np.array([h["cost_pred"] for h in history]) / 2
costs = np.array([h["cost_true"] for h in history]) / 2
error = np.abs((predicts - costs) / costs)
ax.plot(rounds, error, "-", alpha=1, label="Predicted")
ax.set_yscale("log")
ax.set_xlabel("Round")
ax.set_ylabel("Relative Error (Absolute)")
ax.set_title("Prediction Error")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
cmap = plt.get_cmap("viridis").copy()
pc = ax.pcolormesh(i_r, e_r, np.log10(cost/2), cmap=cmap, shading="auto", vmax=-2.4)
fig.colorbar(pc, ax=ax, label="Finesse cost (log scaled)")

n_init_valid = len(xs) - len(history)
init, prop = xs[:n_init_valid], xs[n_init_valid:]
ax.scatter(*init.T, c="white", edgecolors="k", s=10, label="initial")
sc = ax.scatter(*prop.T, c=np.arange(len(prop)), cmap="Reds",
                edgecolors="k", s=30, label="proposals")
fig.colorbar(sc, ax=ax, label="round")
ax.set_xlabel("ITM ROC (m)"); ax.set_ylabel("ETM ROC (m)")

ax.scatter(pso.gbest_x[0], pso.gbest_x[1], marker="*", c='b',s=150, label=f'PSO Best')

ax.legend(loc="upper right")

In [ ]:
print(f"Best cost found by NN: {ys.min()}")
print(f"Best cost found by PSO: {pso.gbest_y[0]}")
print(f"NN Optimal is within {100 * (ys.min() - pso.gbest_y[0]) / pso.gbest_y[0]:.8f} % of the PSO optimal")

In [ ]:
print(ys[:n_init_valid].min())
print(ys.min())
print(ys.min() - pso.gbest_y[0])

In [ ]:
print(calc_cost_perturb([-2575.356, 3788.523]))